# ChainGuard · M4 — Remplir Kibana avec les alertes Bitcoin

**Membre 4 (Laila)** · semaine 3 · prérequis : Elasticsearch démarré (`docker compose -f elastic/docker-compose.m4.yml up -d`).

Le dashboard doit avoir des données avant que la chaîne en direct (listener d'Amira, wallets d'Aya) soit prête. On indexe les transactions de test d'Elliptic au score ≥ 0.3, chacune avec ses 3 features SHAP principales et une phrase d'explication. Les horodatages répartissent les périodes 35 à 49 sur les 7 derniers jours, dans l'ordre : la chronologie Kibana montre donc la dérive.

In [ ]:
import sys; sys.path.append('..')
from collections import Counter
from src.m4.elliptic import load_elliptic
from src.m4.seed import build_btc_alerts, train_btc_model, MODEL_PATH
from src.common.elastic import ES_URL, bulk_index, check, count, setup_index, to_doc

check()                                   # erreur claire si Elasticsearch ne répond pas
print("Elasticsearch OK :", ES_URL)

In [ ]:
E = load_elliptic()
clf = train_btc_model(E)                  # même XGBoost que xgb_none, sauvegardé pour l'API d'Amira
alerts = build_btc_alerts(E, clf, min_score=0.3, days=7)
print(len(alerts), "alertes ; modèle :", MODEL_PATH)
print(Counter(to_doc(a)["severity"] for a in alerts))
to_doc(alerts[0])

In [ ]:
setup_index(reset=True)                   # reset=True : repart d'un index vide (à retirer une fois la démo en place)
n = bulk_index(alerts)
print(n, "alertes indexées ; total dans l'index :", count())

Vérifier dans un terminal :

```
curl "http://localhost:9200/chainguard-alerts/_count"
curl "http://localhost:9200/chainguard-alerts/_search?size=1&pretty"
```

Puis `python elastic/setup_kibana.py` crée la data view et la règle critique. Le dashboard se construit ensuite à la main (voir `elastic/README_kibana.md`).